# Trying to make Charater-Level Language Model

Different from the Word-Level Language Model that has a dictionary of a words that consist each as a token, a Character-Level has a dictionary of a characters that consist each as a token instead.

In [2]:
import json
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from datasets import load_dataset

c:\Users\ACER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training will run on: {device}")

Training will run on: cuda


### Creating dictionary index lookup

In [26]:
# for special token
special_token = ["<START>", "<UNK>", "<PAD>", "<EOS>"]

with open("./input.txt", "r") as file:
    string_stream = file.read()

string_stream = sorted(set(string_stream))


## dictionary making
index_to_char = {}
char_to_index = {}
last_index = 0

for special_token_index, special_token in enumerate(special_token):
    index_to_char[special_token_index] = special_token
    char_to_index[special_token] = special_token_index
    last_index = special_token_index
    
for character_index, character in enumerate(string_stream):
    follow_up_index = character_index + last_index + 1
    index_to_char[follow_up_index] = character
    char_to_index[character] = follow_up_index

with open(".char-to-index.json", "w") as file:
    json.dump(char_to_index, file)

with open("./index-to-char.json", "w") as file:
    json.dump(index_to_char, file)

## Builing Training data

this first batch of training will use a book corpus from hugging face,
the samples is sliced more randomly and doesnt complete a full sentence, that mean i cannot teach the model to output EOS special token.

I'll do that in the second batch of training.

In [147]:
# i grab one massive data from hugging face for addition
book_corpus = load_dataset("rojagtap/bookcorpus")

# grab the first 50000 row of text
book_corpus = book_corpus["train"]["text"][:50000]

x = []
y = []

for line in book_corpus:
    line = line.replace(" .", ".").replace(" ,", ",").replace(" ?", "?").replace(" !", "!").replace(" :", ":")
    line = [char for char in line]
    
    # replace the line with the index
    line = [char_to_index.get(char, char_to_index["<UNK>"]) for char in line]
    
    # prepare x and y
    x_sample = torch.cat((torch.tensor([char_to_index.get("<START>")]), torch.tensor(line)[:-1]), dim=0)
    y_sample = torch.tensor(line)
    
    x.append(x_sample)
    y.append(y_sample)
    
x = nn.utils.rnn.pad_sequence(x, batch_first=True, padding_value=2)
y = nn.utils.rnn.pad_sequence(y, batch_first=True, padding_value=2)

now we have X and Y.
both samples are converted to their index, and are padded to comply the rule of matrix.

for X, it is shifted to the right and the last index are omitted. Also appended a special START token at the first array index.

In [62]:
embedding_count = len(char_to_index)
embedding_dim = 64

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(embedding_count, embedding_dim, 2)
        self.lstm = nn.LSTM(embedding_dim, hidden_size=128, batch_first=True)
        self.linear = nn.Linear(128, embedding_count)
        
    def forward(self, x):
            x = self.embedding(x)
            x, (hn, cn) = self.lstm(x)
            x = self.linear(x)
            return x

model = Model().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=2)
optimizer = optim.Adam(model.parameters(), lr=0.001)

lets do training using Dataloader to slice our massive samples into batches so it doesnt crash.

In [154]:
dataset = TensorDataset(x, y) # type: ignore
dataloader = DataLoader(dataset, 50, shuffle=True)
epochs = 5

for epoch in range(epochs):
    total_loss = 0
    batch_count = 0
    
    for x_batch, y_batch in dataloader:
        # move this batch to CUDA
        x_batch = x_batch.to(device)
        x_batch = x_batch.to(device)
        
        optimizer.zero_grad()
        
        logits = model(x_batch)

        flatten_logits = logits.view(-1, embedding_count).to(device)
        flatten_truth = y_batch.view(-1).to(device)
        
        loss = criterion(flatten_logits, flatten_truth)
        
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        torch.cuda.empty_cache()
        
        total_loss += loss
        batch_count += 1
    
    avg_loss = total_loss / batch_count
    print(f"epoch {epoch}: average loss = {avg_loss}")

epoch 0: average loss = 1.3023011684417725
epoch 1: average loss = 1.3012641668319702
epoch 2: average loss = 1.3001728057861328
epoch 3: average loss = 1.2992786169052124
epoch 4: average loss = 1.298235535621643


i did approximately 40 epochs with loss from 2.2 to 1.29, no oscillation which is perfect. Its now time to test it.

In [43]:
model.eval()

k = 10
p = 0.85
temp = 1.2

def generate(prompt):
    with torch.no_grad():
        prompt = [char_to_index.get(char, char_to_index["<UNK>"]) for char in prompt]
        prompt = torch.tensor(prompt).unsqueeze(0).to(device)

        logits = model(prompt)
        logits = logits[0, -1, :]
        
        logits = logits / temp
        
        topk = torch.topk(logits, k=5)
        softmax = torch.nn.functional.softmax(topk.values, dim=0)

        cum_prob = 0.0
        index_cutoff = 0
        for i, prob in enumerate(softmax):
            if cum_prob >= 0.9:
                index_cutoff = i
                break
                    
            cum_prob += prob
            
        # safety measure bcz if index 0, then it will grab nothing
        index_cutoff = 1 if index_cutoff == 0 else index_cutoff
        
        p_val = softmax[:index_cutoff]
        p_val = p_val / p_val.sum()
        
        p_idx = topk.indices[:index_cutoff]
            
        choice = torch.multinomial(p_val, num_samples=1)
        choice = p_idx[choice]
        
        character_choice = index_to_char.get(choice.tolist()[0])
        
        return character_choice

i created a hyper parameter for top-P top-K and temperature, and a dedicated generate function!

now i can loop easily to generate sentence.

In [64]:
reached_dot = False
prompt = "She is, but "
max_len = 50

while not reached_dot:
    context = prompt[-max_len:]    
    
    result = generate(context)
    prompt += result #type: ignore
    reached_dot = True if result == "." else False
    
print(prompt)

She is, but it is the stop.


result:
- "some " -> "some of the concreece word was a stuff."
- "many " -> "many was a stars."
- "he " -> "he was still stay and stated."
- "those " -> "those discussion was all tourists and stayed at the stomes."
- "World is " -> "World is as it seen anyway, im going to breather, a strange story of the strean."
- "She is, but " -> "She is, but we would have talk."

for a dataset that is only a book corpus, this is good.
The model learn how to spell each character, putting space in the right place, putting comma and dot perfectly, and learned plural and past tense.

### Saving the model

In [334]:
path = "weights.pth"

torch.save(model.state_dict(), path)

Now the model know how to spell something, but instead of asnwering the prompt, its just continuing the story of the prompt. We can now train the model again to teach it not to continue the prompt but to answer it. 
<br>
<br>
what we have right now is the base model, and the next step is called <b>Instruction Fine Tuning</b>.

oh since the datasets row consist of question and answer, i can append special END token too so the AI learn how to stop generating.

In [39]:
instruction_samples = load_dataset("tatsu-lab/alpaca")

x = []
y = []

print(instruction_samples["train"])

for i in range(len(instruction_samples["train"]["instruction"])):
    instruction = instruction_samples["train"]["instruction"][i]
    input = instruction_samples["train"]["input"][i]
    output = instruction_samples["train"]["output"][i]
    
    if input != "":
        continue
    
    base = "User: " + instruction + " Me: " + output
    base = [char_to_index.get(char, char_to_index["<UNK>"]) for char in base]
    
    x_sample = [0] + base
    y_sample = base + [3]
    
    x.append(torch.tensor(x_sample))
    y.append(torch.tensor(y_sample))
    
x = torch.nn.utils.rnn.pad_sequence(x, padding_value=2, batch_first=True)
y = torch.nn.utils.rnn.pad_sequence(y, padding_value=2, batch_first=True)

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 52002
})


i should mix it with the dataset training before but for the sake of time, ill just train it using this only. It will break something that the model already learned but i will take the risk.

In [65]:
model.load_state_dict(torch.load("./weights.pth"))
model.train()
model.to(device)

dataset = TensorDataset(x, y) # type: ignore
dataloader = DataLoader(dataset, 50, shuffle=True)
epochs = 30

# drop the learning rate because we only do fine tuning
optimizer = optim.Adam(model.parameters(), lr=0.00025)

for epoch in range(epochs):
    total_loss = 0
    batch_count = 0
    
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        
        logits = model(x_batch)
        
        flatten_logits = logits.view(-1, embedding_count).to(device)
        flatten_truths = y_batch.view(-1).to(device)
        
        loss = criterion(flatten_logits, flatten_truths)
        
        loss.backward()
        optimizer.step()
        
        torch.cuda.empty_cache()
        
        total_loss += loss
        batch_count += 1
    avg_loss = total_loss / batch_count
    print(f"epoch {epoch}: average loss = {avg_loss}")

epoch 0: average loss = 2.0313549041748047
epoch 1: average loss = 1.6398917436599731
epoch 2: average loss = 1.5365657806396484
epoch 3: average loss = 1.4863213300704956
epoch 4: average loss = 1.4542800188064575
epoch 5: average loss = 1.431160569190979
epoch 6: average loss = 1.4131466150283813
epoch 7: average loss = 1.3984495401382446
epoch 8: average loss = 1.3867634534835815
epoch 9: average loss = 1.3760926723480225
epoch 10: average loss = 1.3668663501739502
epoch 11: average loss = 1.3590341806411743
epoch 12: average loss = 1.3518397808074951
epoch 13: average loss = 1.3449742794036865
epoch 14: average loss = 1.3395336866378784
epoch 15: average loss = 1.3339800834655762
epoch 16: average loss = 1.3284333944320679
epoch 17: average loss = 1.3240388631820679
epoch 18: average loss = 1.3197437524795532
epoch 19: average loss = 1.3158773183822632
epoch 20: average loss = 1.311846375465393
epoch 21: average loss = 1.3087297677993774
epoch 22: average loss = 1.3050793409347534


Testing yay after 71 minutes training

In [90]:
model.eval()

k = 10
p = 0.90
temp = 0.7

reached_dot = False
prompt = "User: Describe a box. Me: "
max_len = 50

while not reached_dot:
    context = prompt[-max_len:]    
    
    result = generate(context)
    prompt += result #type: ignore
    reached_dot = True if result == "." else False
    
print(prompt)

User: Describe a box. Me: The sun should be a policies and the sentence to the species and the products are allowed and solve an internet of a strategy and trained to stay a people are a personal and the starts in the sentence to the story is the stars to the sun should be used to consideration.


<b>Temp: 1.2, </b><br>
<b>result:</b> <br>
"User: what is DNA?" <br>
"Me: The strength in technology and the procedutional and a state of tripding are thouse that cause of the streason ories." <br>
<br>
"User: Develop a plan." <br>
"Me: In the stress of a stability." <br>
<br>
"User: The capital of spain." <br>
"Me: The street of strange and standards, and they can be used to create the streams in the states of the stars to the child apprectricia the process of the potential customer service tasks in the story." <br>
<br>
"User: What is oxygen." <br>
"Me: The strees." <br>
<br>
"User: Describe a box." <br>
"Me: The streaming statement in the company." <br>

---

<b>Temp: 0.7, </b><br>
<b>result:</b> <br>
"User: what is DNA?" <br>
"Me: 1" <br>
<br>
"User: Develop a plan." <br>
"Me: A sentences, someone can be used to consistently to the story and such as the solution of the story are a private and the sentence to the sun should also be used to contraction and such as the sentence and the sentence of a start and the possible to the street." <br>
<br>
"User: The capital of spain." <br>
"Me: The strategies in the conversation in the computing all the change of a sentence in the student of the change of a sun is computer and the strong starts and treatment of the sun service include the presentation are the sentence to the sentence and the sun should be a part of the story of a successful of the streaming sentence and the sun supporting the presentation, and the students are a paramones are contains to the sentence of a strategy to the strength and the sentence the programming in the computer of the communication of sentence is the sun should have a strategy should be considered to the strategies to constructive and and the solar and algorithm that can help to create the sun search and a sentence to the sun statement and the sources is a story and the sun surface to the story of the strategy and the solutions are the perfect of the conserve of the strength and the sentence in the stress of a story and a successful of the characteristics and stars and the second of the community is a state to an example of the string are structure to the summarization is the sentence and the source of a sun is a colors or an engineer to the sentence and the sentence is a state of the sentence and the source and a star and the product with a second of a sentence is an algorithms and the starting a positive sentence and the search of the store of the concept of customer service to the sentence to the sentence and the sun, the presentation of the control and the possible to the string, the provides a large successful of a successful of the sun should have a sentence is a sentence and the sea levels in the sun service in the sun should be an applications and services in the security of the sentence of the security of the solution that can be used to common and a sun in the sun successful of the stress of the company can be used to connective to the strategies that has a product of the strong story of the streets are the story of company that a presing and the sentence is the stars that can be able to confises and contrast the personal and all include an interview of the countries of the sun service in the start of a security of the customer service to the string of the process of a strategy to take a speech and stars of the stars and the sentence to the sun service is allowing the products." <br>
<br>
"User: What is oxygen." <br>
"Me: The sentence is the sun success of the stress in the sentence that can be a product of complete in the stars and the stars to the concentration of customer service to the statement and the process of a service is a story of a strange of strength to a star and the song and strategies are a product of the sources of the story also can be used to complete the progress of the story and stores in a star and the sum of the sun success are the process to the company in an activities are the sentence to the story of the story are also a post on the sun storage and the strong structure of a sentence." <br>
<br>
"User: Describe a box." <br>
"Me: The sun should be a policies and the sentence to the species and the products are allowed and solve an internet of a strategy and trained to stay a people are a personal and the starts in the sentence to the story is the stars to the sun should be used to consideration." <br>

as expected, the model produce gibberish the more it generate. It is a hard cap for LSTM, i get it. Atleast the model know ho to answer and not continuing the prompt. 